In [ ]:
import sys 
import os

%load_ext autoreload
%autoreload 2
SCRIPT_DIR = os.path.dirname(os.path.realpath(__vsc_ipynb_file__))
sys.path.append(os.path.dirname(SCRIPT_DIR))

import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path
from world_wrapper import WorldWrapper


wrapper = WorldWrapper(Path("/home/kyre/repos/minecraft-world-generator/tmp/processed_worlds/cleansed/6659184"))
coords = sorted(wrapper._mca_coords)

INFO - Loading level /home/kyre/repos/minecraft-world-generator/tmp/processed_worlds/cleansed/6673170


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Checking regions: 100%|██████████| 128/128 [00:00<00:00, 1483.58it/s, region=r.-2.5]


In [5]:
print(wrapper.mca_coords)

[(4, 0), (-5, -3), (-6, 1), (5, 1), (1, -6), (0, -4), (-3, 6), (0, 5), (6, 2), (-4, 1), (4, 2), (3, 6), (5, 3), (-6, 3), (2, -5), (2, 4), (-4, -6), (6, 4), (-4, 3), (4, -5), (-6, -4), (5, -4), (5, 5), (-6, 5), (-5, 4), (2, -3), (6, -3), (-4, -4), (-7, 6), (-4, 5), (-1, 2), (-6, -1), (5, -2), (5, -1), (-3, -6), (0, -7), (-6, -2), (-5, 6), (-3, 3), (-1, -5), (-2, -5), (-1, 4), (-2, 4), (3, -6), (3, 3), (-6, 0), (5, 0), (-3, -4), (0, -5), (-3, 5), (1, -4), (1, 5), (6, 1), (-7, 1), (3, -4), (3, 5), (-6, 2), (4, 4), (-5, 1), (5, 2), (-4, -7), (-7, 3), (-4, 2), (3, -2), (-6, -5), (4, -3), (-5, -6), (3, -1), (5, -5), (5, 4), (-6, 4), (4, 6), (-5, 3), (-7, -4), (-4, -5), (-7, 5), (-4, 4), (-6, -3), (5, -3), (-5, -4), (-6, 6), (-5, 5), (0, 4), (-7, -1), (-7, -2), (-1, 5), (-2, 6), (4, 1), (-5, -1), (-5, -2), (-3, -5), (-3, 4), (1, -5), (2, -6), (1, 4), (2, 3), (6, 3), (3, -5), (4, -6), (3, 4), (4, 3), (-5, 0), (1, -3), (2, -4), (2, 5), (-7, 2), (6, -4), (6, 5), (5, -6), (4, -4), (-5, -7), (4, 5

In [7]:
region = wrapper.get_region_volume(-3, -5, use_fast_extractor=True)
print(region.shape)

(512, 512, 348)


In [3]:
print(region.shape)

(512, 512, 134)


In [5]:
import numpy as np

In [7]:
print(len(np.unique(region)))

2285


In [8]:
import numpy as np
print(region[0].shape)

ids, cnts = np.unique(region[0], return_counts=True)
for block_id, count in sorted(zip(ids, cnts), key=lambda x: x[1], reverse=True):
    print(f"Block ID {block_id}: {count} blocks")

(512, 348)
Block ID 0: 81190 blocks
Block ID 4: 43262 blocks
Block ID 457: 25807 blocks
Block ID 380: 4702 blocks
Block ID 461: 3180 blocks
Block ID 5391: 2980 blocks
Block ID 1216: 2507 blocks
Block ID 17: 1621 blocks
Block ID 1317: 1397 blocks
Block ID 1458: 1388 blocks
Block ID 1255: 1370 blocks
Block ID 115: 1335 blocks
Block ID 2391: 1043 blocks
Block ID 41: 944 blocks
Block ID 3445: 687 blocks
Block ID 954: 651 blocks
Block ID 2: 512 blocks
Block ID 929: 499 blocks
Block ID 1927: 365 blocks
Block ID 278: 302 blocks
Block ID 2046: 255 blocks
Block ID 8409: 208 blocks
Block ID 3572: 165 blocks
Block ID 9039: 137 blocks
Block ID 2217: 134 blocks
Block ID 9038: 121 blocks
Block ID 1354: 118 blocks
Block ID 3446: 116 blocks
Block ID 71: 112 blocks
Block ID 1353: 104 blocks
Block ID 727: 81 blocks
Block ID 8935: 76 blocks
Block ID 2045: 62 blocks
Block ID 9023: 51 blocks
Block ID 1281: 44 blocks
Block ID 1823: 43 blocks
Block ID 37: 42 blocks
Block ID 463: 32 blocks
Block ID 124: 31 bl

In [9]:
import numpy as np
from pathlib import Path

# Load the pre-processed RGB LUT generated by visualize_embeddings.py
# This is natively a (vocab_size, 3) array of uint8!
lut_path = "../../tmp/block_embeddings_rgb.npy"
lut = np.load(lut_path)
print(lut.shape)

# 3. Perform the mapping
# ids_cube contains values from 0 to len(lut)-1
# lut[0] = [0, 0,0]
rgb_cube = lut[region]

print(rgb_cube.shape)


(12307, 3)
(512, 512, 348, 3)


In [ ]:
import napari 

# 1. Fix the axis order: (512, 512, 384, 3) -> (384, 512, 512, 3)
# We move the Z-axis (index 2) to the front (index 0)
fixed_cube = rgb_cube

viewer = napari.Viewer(ndisplay=3)

viewer.add_image(
    fixed_cube,
    rendering='iso',
    iso_threshold=0.1,
    blending='additive',
    interpolation3d='nearest',
    interpolation2d='nearest'
)

viewer.dims.ndisplay = 3
napari.run()

INFO - NumExpr defaulting to 16 threads.
INFO - No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'


---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File ~/repos/minecraft-world-generator/.venv/lib/python3.12/site-packages/napari/_qt/threads/status_checker.py:126, in StatusChecker.calculate_status(self=<napari._qt.threads.status_checker.StatusChecker object>)
    122     return
    124 try:
    125     # Calculate the status change from cursor's movement
--> 126     res = viewer._calc_status_from_cursor()
        viewer = Viewer(camera=Camera(center=(255.5, 255.5, 173.5), zoom=1.4843749999999996, angles=(107.78960062099857, -12.604526672760509, -70.41177422574145), perspective=0.0, mouse_pan=True, mouse_zoom=True, orientation=(<DepthAxisOrientation.TOWARDS: 'towards'>, <VerticalAxisOrientation.DOWN: 'down'>, <HorizontalAxisOrientation.RIGHT: 'right'>)), cursor=Cursor(position=(-2.0199238521345535, 782.0140512531468, 469.0969968748253), viewbox=(0, 0), scaled=True, style=<CursorStyle

In [8]:
from skimage import measure

# 1. Generate the surface mesh
# 'level' is the intensity value where the surface will be drawn
input = region[0].copy()
suggested_level = (input.max() + input.min()) / 2
verts, faces, normals, values = measure.marching_cubes(input, level=1)

try:
    viewer = napari.current_viewer()
    if viewer is None:
        viewer = napari.Viewer()
except:
    viewer = napari.Viewer()


# 2. Add to napari as a Surface layer
# We pass a tuple of (vertices, faces, values)
viewer.add_surface((verts, faces, values), name='Surface Mesh')

ValueError: Input volume should be a 3D numpy array.

In [ ]:
import pyvista as pv 

import pyvista as pv
import numpy as np
from scipy.ndimage import binary_dilation

import pyvista as pv
import numpy as np
from scipy.ndimage import binary_dilation

def render_minecraft_iso(data, filename="minecraft_rotated.png", rotation_angle=45):
    # 1. Surface Extraction
    is_solid = np.any(data > 0, axis=-1)
    is_air = ~is_solid
    surface_mask = is_solid & binary_dilation(is_air)

    z, y, x = np.where(surface_mask)
    points = np.column_stack((x, y, z)) 
    surface_colors = data[surface_mask]

    # 2. Geometry
    point_cloud = pv.PolyData(points)
    cube = pv.Cube()
    blocks = point_cloud.glyph(geom=cube, scale=False)

    # 3. Manual Data Injection
    repeated_colors = np.repeat(surface_colors, 6, axis=0)
    blocks.cell_data["colors"] = repeated_colors

    # 4. Setup Plotter
    pl = pv.Plotter(off_screen=True, lighting=None)
    pl.background_color = 'black'
    pl.enable_ssao(radius=2.0, bias=0.5) 
    pl.enable_3_lights()
    
    pl.add_mesh(
        blocks, 
        scalars="colors", 
        rgb=True, 
        preference='cell',
        smooth_shading=False,
        ambient=0.2,
        diffuse=0.8
    )

    # 5. Fixed Camera & Rotation Logic
    pl.enable_parallel_projection()
    
    # Calculate exact center
    center = np.array([data.shape[1]/2, data.shape[0]/2, data.shape[2]/2])
    
    # Change these multipliers to rotate:
    # (1, 1, 1) -> Front-Right
    # (-1, 1, 1) -> Front-Left
    # (-1, -1, 1) -> Back-Left
    # (1, -1, 1) -> Back-Right
    iso_vector = np.array([1, 1, 0.8]) # Adjusted for a better "North-West" view
    dist = max(data.shape) * 2
    
    pl.camera.position = center + (iso_vector * dist)
    pl.camera.focal_point = center
    pl.camera.up = (0, 0, 1) 
    
    pl.reset_camera()
    pl.camera.zoom(1.1)

    # 6. Finalize
    pl.screenshot(filename)
    pl.close()
    print(f"Rotated render saved to {filename}")
# render_minecraft_iso(your_np_array)
    
render_minecraft_iso(rgb_cube)